In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_ticketing_request_ssp(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        # "TicketingRequestId string,"
        # "TicketingRequestKey string,"
        # "TicketingRequestCode string,"
        # "TicketingRequestGuid string,"
        # "SourceSystem string,"
        # "IngestionTimestamp timestamp,"
        # "BatchId string,"
        # "WorkflowDataKey int,"
        # "UserType string,"
        # "UserGuid string,"
        # "EmailAddress string,"
        # "PhoneNumber string,"
        # "PartyType string,"
        # "SourcePortfolioNumber string,"
        # "SourcePortfolioName string,"
        # "AgreementKey string,"
        # "TaxRegistrationNumber string,"
        # "RequestCategoryTypeKey string,"
        # "ComplaintSubject string,"
        # "Version int,"
        # "CreatedDate timestamp,"
        # "ModifiedDate timestamp,"
        # "WorkflowApplicationStatus string,"
        # "WorkflowApplicationKey string,"
        # "WorkflowApplicationTitle string,"
        # "WorkflowApplicationName string,"
        # "RequestData string,"
        # "ResponseData string,"
        # "WorkflowCreatedDate timestamp,"
        # "WorkflowModifiedDate timestamp,"
        # "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.TicketingRequest"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "ssp",
            )        
        else:
            logging.info("Tables do not exist, no need to delete entries for business date!")

        # ================== ticketingrequest ==================

        ticketingrequest_extra_filters = None

        if ticketingrequest_extra_filters is not None:
            ticketingrequest_extra_filters.append(date_filter)
            ticketingrequest_final_filters = " AND ".join(
                ticketingrequest_extra_filters
            )
        else:
            ticketingrequest_final_filters = date_filter

        ticketingrequest_selected_fields = [
            "workflowdataid",
            "id",
            "ticketingrequestguid",
            "workflowdataid",
            "userguid",
            "email",
            "phone",
            "debtortype",
            "portfolionumber",
            "portfolioname",
            "contractnumber",
            "vatnumber",
            "requestcategorytypeid",
            "complaintsubject",
            "version",
            "createdon",
            "modifiedon",
            "businessdate",
        ]

        ticketingrequest = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "ssp_ticketingrequest",
            ticketingrequest_final_filters,
            ticketingrequest_selected_fields,
        )




        # ================== RequestCategoryType ==================

        requestcategorytype_extra_filters = None

        if requestcategorytype_extra_filters is not None:
            requestcategorytype_extra_filters.append(date_filter)
            requestcategorytype_final_filters = " AND ".join(
                requestcategorytype_extra_filters
            )
        else:
            requestcategorytype_final_filters = date_filter

        requestcategorytype_selected_fields = [
            "Name",
            "RequestCategoryTypeID",
            "businessdate",
        ]

        requestcategorytype = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "ssp_requestcategorytype",
            requestcategorytype_final_filters,
            requestcategorytype_selected_fields,
        )

        # ================== workflowdata ==================

        workflowdata_extra_filters = None
        if workflowdata_extra_filters is not None:
            workflowdata_extra_filters.append(date_filter)
            workflowdata_final_filters = " AND ".join(workflowdata_extra_filters)
        else:
            workflowdata_final_filters = date_filter
        workflowdata_selected_fields = [
            "id",
            "workflowapplicationstatus",
            "workflowapplicationid",
            "workflowapplicationtitle",
            "workflowapplicationname",
            "requestdata",
            "responsedata",
            "createdon",
            "modifiedon",
        ]
        workflowdata = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "ssp_workflowdata",
            workflowdata_final_filters,
            workflowdata_selected_fields,
        )

        # ========================== Joins ==========================

        ticketingrequest_workflowdata_join_condition = (
            ticketingrequest["workflowdataid"] == workflowdata["id"]
        )

        ticketingrequest_requestcategorytype_join_condition = (
            ticketingrequest["RequestCategoryTypeID"] == requestcategorytype["RequestCategoryTypeID"]
        )

        target_table = ticketingrequest.join(
            workflowdata,
            ticketingrequest_workflowdata_join_condition,
            "left",
        ).join(
            requestcategorytype,
            ticketingrequest_requestcategorytype_join_condition,
            "left",
        )

        primary_key_fields = [
            ticketingrequest.id,
            ticketingrequest.ticketingrequestguid,
            lit("ssp"),
        ]

        target_table = (
            target_table.withColumn(
                "TicketingRequestId", sha2(concat_ws(";", *primary_key_fields), 256)
            )
            .withColumn(
                "TicketingRequestKey",
                concat_ws(
                    ";",
                    ticketingrequest.id,
                    ticketingrequest.ticketingrequestguid,
                ),
            )
            .withColumn(
                "UserType",
                when(ticketingrequest.userguid.isNull(), lit("guest")).otherwise(
                    lit("member")
                ),
            )
        )

        result_df = target_table.select(
            target_table.TicketingRequestId.cast("string").alias("TicketingRequestId"),
            target_table.TicketingRequestKey.cast("string").alias(
                "TicketingRequestKey"
            ),
            ticketingrequest.id.cast("string").alias("TicketingRequestCode"),
            ticketingrequest.ticketingrequestguid.cast("string").alias(
                "TicketingRequestGuid"
            ),
            lit("ssp").cast("string").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            ticketingrequest.workflowdataid.cast("int").alias("WorkflowDataKey"),
            target_table.UserType.cast("string").alias("UserType"),
            ticketingrequest.userguid.cast("string").alias("UserGuid"),
            ticketingrequest.email.cast("string").alias("EmailAddress"),
            ticketingrequest.phone.cast("string").alias("PhoneNumber"),
            ticketingrequest.debtortype.cast("string").alias("PartyType"),
            ticketingrequest.portfolionumber.cast("string").alias(
                "SourcePortfolioNumber"
            ),
            ticketingrequest.portfolioname.cast("string").alias("SourcePortfolioName"),
            ticketingrequest.contractnumber.cast("string").alias("AgreementKey"),
            ticketingrequest.vatnumber.cast("string").alias("TaxRegistrationNumber"),
            ticketingrequest.requestcategorytypeid.cast("string").alias(
                "RequestCategoryTypeKey"
            ),
            ticketingrequest.complaintsubject.cast("string").alias("ComplaintSubject"),
            ticketingrequest.version.cast("int").alias("Version"),
            ticketingrequest.createdon.cast("timestamp").alias("CreatedDate"),
            ticketingrequest.modifiedon.cast("timestamp").alias("ModifiedDate"),
            workflowdata.workflowapplicationstatus.cast("string").alias(
                "WorkflowApplicationStatus"
            ),
            workflowdata.workflowapplicationid.cast("string").alias(
                "WorkflowApplicationKey"
            ),
            workflowdata.workflowapplicationtitle.cast("string").alias(
                "WorkflowApplicationTitle"
            ),
            workflowdata.workflowapplicationname.cast("string").alias(
                "WorkflowApplicationName"
            ),
            workflowdata.requestdata.cast("string").alias("RequestData"),
            workflowdata.responsedata.cast("string").alias("ResponseData"),
            workflowdata.createdon.cast("timestamp").alias("WorkflowCreatedDate"),
            workflowdata.modifiedon.cast("timestamp").alias("WorkflowModifiedDate"),
            requestcategorytype.Name.cast("string").alias("RequestCategoryType"),
            ticketingrequest.businessdate.alias("BusinessDate"),
        )

        # ========================== Append ==========================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_ticketing_request_ssp(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)